### <h1 align="center">**Polynomial CURVE FIT function algorithm for individual data for the pole_to_chromosome distance**</h1> 

This script fits a polynomial curve to the **INDIVIDUAL** experiment (n-value) of each cell type of the *C. elegans* embryo in batch process. This uses the polynomial curve-fitting function where $y = p_n * x^n + p_{n-1} * x^{n-1} + ... + p_1 * x + p_0$. The goal for fitting a mathematical function to the individual data is to determine the **Initial pole-to-chromosome length**, **Final pole-to-chromosome length**, **Rate of distance reduction** and the **ratio of the final length to the initial length** (normalization) at the required time point.

`INPUT FILES` 

The csv files of each cell type containing the pole-to-chromosome distance data of different experiments (n-value). 

The table below shows the table structure of each .csv file. The headers of each table should show the time series measurement value from each observation which starts with prefix, **Exp**, and other statistical values from the observations.

|Exp00 | Exp01 | Exp02 | Exp03 | Exp04 | Exp05 | mean | std | n | SE | time |
| :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----:| :-----: | :-----: |
| `num` | `num` | `num` | `num` | `num` | `num` | `num` | `num` | `num`| `num` | `num` |
| `...` | `...` | `...` | `...` | `...` | `...` | `...` | `...` | `...`| `...` | `...` |

`OUTPUT FILES` 
* **FIRST_GROUP_OUTPUT_FILES**: The *.png* files of the fitted plot of each cell type. 

* **SECOND_GROUP_OUTPUT_FILE**: A *.csv* file containing the **Initial length (µm)**, the **Final length (µm)**, **Rate of distance reduction (µm/minute)** and **Ratio (final length/initial length)** of each of the cell type. 


In [1]:
# library packages
import os
import warnings
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from itertools import zip_longest
from scipy.stats import linregress
from numpy import exp, linspace, random, arange
from scipy.optimize import curve_fit, least_squares

In [2]:
# input folder
folder = r'D:\data\Analysis Data\python_analysis\input'

# output folder
save_files = r'D:\data\Analysis Data\python_analysis\output'

In [3]:
# new DataFrame to append the new generated table 
fit_Result = pd.DataFrame()

# read out individual files and compute for different operations 
for file in os.scandir(folder):
    df = pd.read_csv(file)
    
    # create a new DataFrame 
    Exp_Column = df.loc[:, df.columns.str.startswith('Exp')]
    mean_column = df.loc[:, df.columns.str.startswith('mean')]
    time_column = df.loc[:, df.columns.str.startswith('time')]
    
    # define the start and end time points to compute
    x_start = np.where(time_column >= 0)[0][0]
    x_end = np.where((time_column <= 154.5) & (time_column == 154.5))[0][0] + 1
    
    # compute the time frames for column values
    new_x = time_column[x_start:x_end]
    new_exp = Exp_Column[x_start:x_end]
    new_mean = mean_column[x_start:x_end]
    
    # create a new DataFrame
    df_Table = pd.concat([new_x, new_exp, new_mean], axis=1)
    
    '''
    drop all the rows were n-value is less than 3 the mean and the time 
    columns are included, ie, the number of rows to be computed should be >= 3
    '''
    newTable = df_Table.dropna(thresh=3)   
    
    # define a polynomial function to fit into the mean values for each cell type
    def polynomial(x, *coeffs):
        return np.polyval(coeffs, x)
    
    # ignore warning
    warnings.filterwarnings("ignore")
    
    # define the plot dimension 
    plt.figure(figsize=(5,5))
    
    # iterate over every column in each DataFrame and plot
    for i_col in newTable.columns[1:-1]:
        
        try:
            NaN_table = newTable[['time', i_col]]        
            plot_table = NaN_table.dropna()

            # x and y variables
            x = plot_table['time']
            y = plot_table[i_col]

            # define the degree of polynomial which is the highest power of x in the given parameters to increase fitting accuracy 
            degree = min(6, len(x) - 1)
            
            # compute the initial guess
            initial_guess = np.ones(degree + 1)

            # summarize the parameter
            popt, pcov = curve_fit(polynomial, x, y, p0=initial_guess)

            ''' 
            Define a sequence of inputs between the smallest and largest known inputs and define the fit. 
            Let the maximum input assume the maximum values of x. 
            '''
            x_fit = np.arange(x.min(), x.max()+0.01, 0.01)
            y_fit = polynomial(x_fit, *popt)

            # primary plot
            ax1 = sns.scatterplot(x=x, y=y, alpha=0.2)

            # fit on each plot 
            sns.lineplot(x=x_fit, y=y_fit, alpha = 1, ax=ax1)

            # add the desired features on the plot
            ax1.set_xlabel('time [s]', fontsize= 18)
            ax1.set_ylabel('distance [μm]', fontsize= 18)
            plot_title = (file.name).split('.')[0]
            ax1.axes.set_title(plot_title, fontsize= 20, fontweight='bold')

            # save plots
            plotfile = (file.name).split('.')[0] + '.png'
            plt.savefig(os.path.join(save_files, plotfile), dpi=300)

            # Calculate the derivative (rate of speed) at time = 0
            coeffs_derivative = np.polyder(popt)
            velocity_at_time_zero = np.polyval(coeffs_derivative, 0)

            # convert the velocity of distance reduction from µm/s to µm/min by multiplying it by 60
            distance_reduction_rate = velocity_at_time_zero * 60

            '''create a new dictionary for each parmeter and add to the dataframe fit_Result'''
            # compute for the initial and the final length from the curve
            initial_length = y_fit[0]
            final_length = y_fit[-1]    
            ratio_final_vs_initial = final_length / initial_length # Normailze to the Initial length to get the relative distance difference
            
            parameters = {'Rate of distance reduction (µm/min)': distance_reduction_rate, 
                          'Initial length (µm)': initial_length, 
                          'Final length (µm)': final_length, 
                          'Ratio (final length / initial length)': ratio_final_vs_initial}

            parameters_df = pd.DataFrame.from_dict(parameters, orient='index', columns=[file.name.split('.')[0]])
            fit_Result = pd.concat([fit_Result, parameters_df], axis=1)
            
        except Exception:
            pass
        
# close all plots open windows
plt.close('all') 

# transpose the table
fit_Result_transpose = (fit_Result).T

# assign header to the index column
fit_Result_transpose.index.names = ['Cells']
    
# save the table, fit_Result, to a csv file
fit_Result_transpose.to_csv(os.path.join(save_files, 'Fit_Result_pole_chromosome.csv'), encoding='utf-8')